## Value of GCR Projects

Based on Tarsney's "Epistemic challenge to longtermsim" paper from 2020. Builds off of it to get numeric computations of the value of the future in short-term periods (next 500 years), when the time-of-perils assumption holds. Then does explicit integration of the value of the future once x-risk levels stabilize and the earth reaches a roughly constant value. 

#### Set-up

In [39]:
import numpy as np
import pandas as pd
from math import exp
import random

### Parameters

## Intervention

In [40]:
M = 10**6
B = 10**9
budget = 10*M # the donor's budget

n_sims = 3 # number of MC simulations to run

### periods of value

In [41]:
periods_value = [0, 5, 10, 20, 100, 500] # these are the endpoints of the time "buckets" that we are considering. We will find the expected value of the 
# future between 0-5 years from now, 5-10 years from now, ..., 100-500 years from now, and 500+ years from now

## Variables

#### Risk trajectory parameters

In [42]:
# Risk trajectory parameters - see get_annual_risk_level() below to modify
cumulative_risk_100_yrs = np.array([0.1, 0.05, 0.25])
year_max_risk = np.array([15, 10, 5])
year_risk_1pct_max = np.array([300, 200, 300]) # year when the Time of Perils is over (risk is down to 1% of its peak)

# long-run annual x-risk level once x-risk stabilizes. Most important parameter
r_inf = np.array([10**(-7), 10**-5, 10**-10])

# when heat death of the universe occurs
T_h = np.array([10**14] * n_sims)

#### Intervention effect parameters

In [43]:
# Intervention effect parameters
year_effect_starts = np.array([0, 3, 2])
bp_reduction_per_bn = np.array([1, -1, 10])
rel_rr_from_int = bp_reduction_per_bn * 0.0001 / B * budget
persistence_effect = np.array([20, 10, 30])

#### Growth rate parameters and function

In [44]:
initial_value = np.array([8*10**9, 2*10**9, 5*10**10]) # initial value of Earth

# period of logistic growth
year_logistic_growth_starts = np.array([0]*n_sims) # assume we're in logistic growth now. 
rate_growth = np.array([0.01, 0.005, 0.02]) # rate of value growth on Earth
carrying_capacity = np.array([2, 1.5, 5])*initial_value # value carrying capacity in multiples of current value
print(carrying_capacity)

[1.6e+10 3.0e+09 2.5e+11]


In [45]:
# period of cubic growth
# from Tarsney 2020 - Epistemic challenge to longtermism
cubic_growth = np.array([False, True, True]) # will we do stellar settlement?
T_c = np.array([300, 1000, 600]) # years from now when cubic growth starts 
r_g = 1.3e5 # radius of the galaxy
s = np.array([0.01, 0.001, 0.1]) # speed of us settling the stars, in multiples of the speed of light
d_g = 2.2e-5 # density of stars in Milky way
d_s = 2.9e-9 # density of stars in Virgo supercluster

In [46]:
### define simplified parameteres for cubic growth function
pi = np.pi
b2 = r_g * (d_g - d_s)
T_s = r_g / s
# v_s, a1, a2 computed later once earth value at T_c is available

### Compute Risk Trajectory

In [ ]:
def _solve_r_max(cumulative_risk, year_max_risk, year_risk_1pct_max, r_inf, n_years=100):
    """Numerical bisection to find r_max that reproduces cumulative_risk.
    Accounts for actual Gaussian shape, peak year, and background risk r_inf.
    Matches gcr_model.py implementation."""
    sigma = year_risk_1pct_max / 3.0
    t = np.arange(n_years + 1, dtype=float)
    def _cum(r_max_vec):
        gaussian = np.exp(-0.5 * ((t[:, None] - year_max_risk[None, :]) / sigma[None, :]) ** 2)
        annual = np.clip(r_inf[None, :] + r_max_vec[None, :] * gaussian, 0.0, 1.0)
        return 1.0 - np.prod(1.0 - annual, axis=0)
    n = len(cumulative_risk)
    lo = np.zeros(n)
    hi = np.ones(n)
    for _ in range(60):  # 2^-60 precision
        mid = 0.5 * (lo + hi)
        cum_mid = _cum(mid)
        hi = np.where(cum_mid < cumulative_risk, hi, mid)
        lo = np.where(cum_mid < cumulative_risk, mid, lo)
    return 0.5 * (lo + hi)

r_max = _solve_r_max(cumulative_risk_100_yrs, year_max_risk, year_risk_1pct_max, r_inf)
print(r_max)
sd_risk = year_risk_1pct_max / 3

def get_annual_risk_level(t, I):
    gaussian_no_int = r_max * np.exp(-((t-year_max_risk)**2/(2*sd_risk**2)))
    r_t_no_int = r_inf + gaussian_no_int
    if I == 1:
        gaussian_int = r_max*(1- rel_rr_from_int)*np.exp(-((t-year_max_risk)**2/(2*sd_risk**2)))
        r_t_int = r_inf + gaussian_int

        # when intervention is active
        in_effect = (t >= year_effect_starts) & (t <= persistence_effect + year_effect_starts)
        r_t = np.where(in_effect, r_t_int, r_t_no_int)
    else:
        r_t = r_t_no_int

    return r_t

def get_p_survival_vec(risk_2d):
    """risk_2d shape (T, n_sims). Returns (T, n_sims) cumulative survival."""
    return np.cumprod(1 - risk_2d, axis=0)

### Figure out how many years until approximately constant risk and value

In [48]:
def get_year_of_const_risk(I):
    """Returns array of shape (n_sims,) with the year risk becomes ~constant."""
    result = np.full(n_sims, 10000)
    converged = np.zeros(n_sims, dtype=bool)
    start = int(np.max(year_max_risk)) + 1
    for t in range(start, 10000):
        r_t = get_annual_risk_level(t, I)
        newly = (~converged) & (np.abs(r_t - r_inf) / r_inf < 0.01)
        result[newly] = t
        converged |= newly
        if converged.all():
            break
    return result

def value_level_logistic(t):
    """t is scalar, returns array of shape (n_sims,)."""
    return carrying_capacity / (1 + (carrying_capacity - initial_value) / initial_value
                                * np.exp(-rate_growth * t))

def get_year_logistic_ends():
    """Returns array of shape (n_sims,)."""
    result = np.full(n_sims, 10000)
    converged = np.zeros(n_sims, dtype=bool)
    start = int(np.max(year_logistic_growth_starts))
    for t in range(start, 10000):
        v_t = value_level_logistic(t)
        newly = (~converged) & (np.abs(v_t) >= 0.99 * carrying_capacity)
        result[newly] = t
        converged |= newly
        if converged.all():
            break
    return result

year_logistic_ends = get_year_logistic_ends()

def get_year_const_value_and_risk(I, to_print=False):
    """Returns array of shape (n_sims,)."""
    yr = get_year_of_const_risk(I)
    yl = get_year_logistic_ends()
    if to_print:
        print(f"Year constant risk (per sim): {yr}")
        print(f"Year logistic growth ends (per sim): {yl}")
    return np.maximum(yr, yl)

year_const_risk_value = get_year_const_value_and_risk(0, True)
print(year_const_risk_value)

Year constant risk (per sim): [548 291 670]
Year logistic growth ends (per sim): [460 781 300]
[548 781 670]


#### Get value functions

In [49]:
def determine_num_years(periods):
    """Returns array of shape (n_sims,) â€” per-sim number of years to simulate."""
    y1 = get_year_const_value_and_risk(1)
    y0 = get_year_const_value_and_risk(0)
    return np.maximum(np.maximum(y1, y0), periods[-1]).astype(int)

### Generate risk, value, survival probabilities numerically

#### Time array

In [50]:
num_years_per_sim = determine_num_years(periods_value)  # (n_sims,)
max_num_years = int(np.max(num_years_per_sim))
years_arr = np.arange(max_num_years + 1)
print(f"num_years_per_sim = {num_years_per_sim}")
print(f"max_num_years = {max_num_years}")

# y_const per sim (used for value computation)
y_const_value = get_year_const_value_and_risk(0, to_print=True)
print(years_arr[-10:])

num_years_per_sim = [548 781 670]
max_num_years = 781
Year constant risk (per sim): [548 291 670]
Year logistic growth ends (per sim): [460 781 300]
[772 773 774 775 776 777 778 779 780 781]


In [51]:
def get_earth_value(t, y_const_value):
    """t scalar, y_const_value array (n_sims,). Returns (n_sims,)."""
    v_logistic = value_level_logistic(t)
    v_const = 0.99 * carrying_capacity
    earth_value = np.where(t < y_const_value, v_logistic, v_const)
    return earth_value

# Compute v_s as earth value at time of star settlement
# assumes that each planet gets earth-like value
# you can change this to be different, e.g. 1/10th the value of the earth
v_s = np.array([get_earth_value(int(T_c[i]), y_const_value)[i] for i in range(n_sims)])
a1 = 4 / 3 * pi * d_g * v_s * s**3
a2 = 4 / 3 * pi * d_s * v_s * s**3
print(f"v_s (value per star at T_c): {v_s}")

v_s (value per star at T_c): [1.52411860e+10 2.97000000e+09 2.49993856e+11]


In [52]:
def get_value_stars_settled(t):
    """t scalar. Returns array (n_sims,)."""
    in_mw = a1 * np.maximum(t - T_c, 0)**3
    beyond_mw = a2 * np.maximum(t - T_c, 0)**3 + b2
    v_stars = np.where(
        cubic_growth & (t >= T_c),
        np.where(t <= T_s, in_mw, beyond_mw),
        0.0
    )
    return v_stars

def get_total_value_level(t, y_const_value):
    # value of civilization conditional on surviving and cubic growth
    v_earth = get_earth_value(t, y_const_value)
    v_stars = get_value_stars_settled(t)
    v_t = v_earth + v_stars
    return v_t

#### Risk arrays

In [53]:
def build_risk_array(I):
    """Returns shape (num_years+1, n_sims)."""
    return np.array([get_annual_risk_level(t, I) for t in years_arr])

# with intervention
r_array_1 = build_risk_array(1)
# without intervention
r_array_0 = build_risk_array(0)

print(r_array_1[:10])
print(r_array_1[-10:])

print(r_array_0[:10])
print(r_array_0[-10:])

[[0.00143705 0.00070975 0.00395936]
 [0.00143914 0.00071125 0.00396114]
 [0.00144108 0.00071259 0.00396248]
 [0.00144288 0.00071378 0.00396348]
 [0.00144454 0.00071481 0.00396407]
 [0.00144606 0.00071568 0.00396427]
 [0.00144743 0.00071639 0.00396407]
 [0.00144867 0.00071695 0.00396348]
 [0.00144975 0.00071735 0.00396248]
 [0.00145069 0.00071759 0.0039611 ]]
[[1.00000001e-07 1.00000000e-05 1.00000666e-10]
 [1.00000000e-07 1.00000000e-05 1.00000617e-10]
 [1.00000000e-07 1.00000000e-05 1.00000571e-10]
 [1.00000000e-07 1.00000000e-05 1.00000529e-10]
 [1.00000000e-07 1.00000000e-05 1.00000490e-10]
 [1.00000000e-07 1.00000000e-05 1.00000453e-10]
 [1.00000000e-07 1.00000000e-05 1.00000420e-10]
 [1.00000000e-07 1.00000000e-05 1.00000389e-10]
 [1.00000000e-07 1.00000000e-05 1.00000360e-10]
 [1.00000000e-07 1.00000000e-05 1.00000333e-10]]
[[0.00143705 0.00070975 0.00395936]
 [0.00143914 0.00071125 0.00396114]
 [0.00144108 0.00071259 0.00396252]
 [0.00144288 0.00071378 0.00396351]
 [0.00144454 0

#### Survival arrays

In [54]:
# probability of surviving each year, with the intervention and without it
survival_arr_1 = get_p_survival_vec(r_array_1)
survival_arr_0 = get_p_survival_vec(r_array_0)
# effect of the intervention on the probability of surviving 
diff_in_survival = survival_arr_1 - survival_arr_0   # (num_years+1, n_sims)

print(survival_arr_1[:10])
print(survival_arr_1[-10:])

print(survival_arr_0[:10])
print(survival_arr_0[-10:])

print(diff_in_survival[:10])
print(diff_in_survival[-10:])

print(f"\nRisk array shape:     {r_array_1.shape}")
print(f"Survival array shape: {survival_arr_1.shape}")
print(f"\nSample risk[0]:      {r_array_1[0]}")
print(f"Sample survival[-1]: {survival_arr_1[-1]}")
print(f"Sample diff surv[-1]:{diff_in_survival[-1]}")

[[0.99856295 0.99929025 0.99604064]
 [0.99712588 0.99857951 0.99209519]
 [0.99568894 0.99786793 0.98816403]
 [0.99425228 0.99715568 0.98424747]
 [0.99281604 0.9964429  0.98034584]
 [0.99138037 0.99572977 0.97645949]
 [0.98994541 0.99501644 0.97258873]
 [0.98851131 0.99430306 0.9687339 ]
 [0.98707822 0.9935898  0.96489531]
 [0.98564627 0.99287681 0.96107326]]
[[0.81484697 0.92841487 0.59488623]
 [0.81484688 0.92840558 0.59488623]
 [0.8148468  0.9283963  0.59488623]
 [0.81484672 0.92838701 0.59488623]
 [0.81484664 0.92837773 0.59488623]
 [0.81484656 0.92836845 0.59488623]
 [0.81484648 0.92835916 0.59488623]
 [0.81484639 0.92834988 0.59488623]
 [0.81484631 0.9283406  0.59488623]
 [0.81484623 0.92833131 0.59488623]]
[[0.99856295 0.99929025 0.99604064]
 [0.99712588 0.99857951 0.99209519]
 [0.99568894 0.99786793 0.98816399]
 [0.99425227 0.99715568 0.98424739]
 [0.99281603 0.99644291 0.98034572]
 [0.99138036 0.99572977 0.97645933]
 [0.9899454  0.99501644 0.97258854]
 [0.9885113  0.99430306 0.

### Value arrays

In [55]:
def build_value_array():
    """Returns shape (max_num_years, n_sims)."""
    return np.array([get_total_value_level(t, y_const_value) for t in range(max_num_years)])

value_array = build_value_array()
print(f"Value array shape:    {value_array.shape}")
print(value_array[:10])
print(value_array[-10:])


Value array shape:    (781, 3)
[[8.00000000e+09 2.00000000e+09 5.00000000e+10]
 [8.03999967e+09 2.00333055e+09 5.08048020e+10]
 [8.07999733e+09 2.00665552e+09 5.16192147e+10]
 [8.11999100e+09 2.00997488e+09 5.24432455e+10]
 [8.15997867e+09 2.01328860e+09 5.32768981e+10]
 [8.19995834e+09 2.01659665e+09 5.41201723e+10]
 [8.23992803e+09 2.01989901e+09 5.49730641e+10]
 [8.27988572e+09 2.02319566e+09 5.58355654e+10]
 [8.31982944e+09 2.02648656e+09 5.67076643e+10]
 [8.35975720e+09 2.02977169e+09 5.75893444e+10]]
[[1.58400000e+10 2.96857231e+09 3.62693761e+11]
 [1.58400000e+10 2.96872743e+09 3.64726545e+11]
 [1.58400000e+10 2.96888178e+09 3.66783105e+11]
 [1.58400000e+10 2.96903538e+09 3.68863578e+11]
 [1.58400000e+10 2.96918823e+09 3.70968102e+11]
 [1.58400000e+10 2.96934034e+09 3.73096816e+11]
 [1.58400000e+10 2.96949170e+09 3.75249858e+11]
 [1.58400000e+10 2.96964232e+09 3.77427365e+11]
 [1.58400000e+10 2.96979220e+09 3.79629478e+11]
 [1.58400000e+10 2.96994136e+09 3.81856333e+11]]


## Get Intervention Value

In [56]:
## we know that the arrays above will run until both risk and earth's value are constant
## We thus just need to know a) whether there will be cubic growth, and b) when it will/has started
## to be able to integrate to the heat death. 
# Scenarios: 
# 1) No cubic growth, just constant earth value to heat death
# 2) Cubic growth, starts after the risk is constant 
# 3) Cubic growth, starts before the risk is constant  

def get_conditional_future_value_on_earth(n_years):
    v_const = 0.99 * carrying_capacity
    earth_V = v_const / r_inf * (1 - np.exp(-r_inf * (T_h - n_years))) # integral over the future value of earth, discounted each year by p(survival) which 
    # is modeled using exponential distribution p(Survival, t) = exp(-r_inf*t)
    return earth_V

def get_conditional_future_value_stars_to_Ts2():
    ## use this if cubic growth starts after risk is constant
    conditional_V_until_Ts = a1 / r_inf * (
        6 / r_inf**3
        - np.exp(-r_inf * (T_s - T_c)) * (
            (T_s - T_c)**3
            + 3 / r_inf * (T_s - T_c)**2
            + 6 / r_inf**2 * (T_s - T_c)
            + 6 / r_inf**3
        )
    )
    return conditional_V_until_Ts


def get_conditional_future_value_stars_to_Ts3(n_years):
    # use this if cubic growth starts before risk is approximately constant
    conditional_V_until_Ts = a1 / r_inf * (
        np.exp(-r_inf * (n_years - T_c)) * (
            (n_years - T_c)**3
            + 3 / r_inf * (n_years - T_c)**2
            + 6 / r_inf**2 * (n_years - T_c)
            + 6 / r_inf**3
        )
        - np.exp(-r_inf * (T_s - T_c)) * (
            (T_s - T_c)**3
            + 3 / r_inf * (T_s - T_c)**2
            + 6 / r_inf**2 * (T_s - T_c)
            + 6 / r_inf**3
        )
    )
    return conditional_V_until_Ts

def get_conditional_future_value_stars_to_Th3():
    # use this if cubic growth starts before risk is approximately constant
    conditional_V_after_Ts = (
        a2 / r_inf * (
            np.exp(-r_inf * (T_s - T_c)) * (
                (T_s - T_c)**3
                + 3 / r_inf * (T_s - T_c)**2
                + 6 / r_inf**2 * (T_s - T_c)
                + 6 / r_inf**3
            )
            - np.exp(-r_inf * (T_h - T_c)) * (
                (T_h - T_c)**3
                + 3 / r_inf * (T_h - T_c)**2
                + 6 / r_inf**2 * (T_h - T_c)
                + 6 / r_inf**3
            )
        )
        + b2 / r_inf * (
            np.exp(-r_inf * (T_s - T_c))
            - np.exp(-r_inf * (T_h - T_c))
        )
    )
    return conditional_V_after_Ts

def get_intervention_shortterm_value(diff_surv, v_arr, n_years):
    """n_years: array (n_sims,). All dict values are arrays of shape (n_sims,)."""
    ev_dict = {}
    for i in range(len(periods_value) - 1):
        lo = periods_value[i]
        hi = periods_value[i + 1]
        ev_i = np.sum(v_arr[lo:hi] * diff_surv[lo:hi], axis=0)
        ev_dict[f"{lo} to {hi}"] = ev_i

    # Last period: periods_value[-1] to n_years[j], which varies per sim.
    # Compute up to max_num_years but mask out years beyond each sim's n_years.
    last = periods_value[-1]
    if np.any(n_years > last):
        time_idx = np.arange(last, max_num_years)[:, None]  # (T_remaining, 1)
        mask = time_idx < n_years[None, :]                   # (T_remaining, n_sims)
        contributions = v_arr[last:max_num_years] * diff_surv[last:max_num_years]
        ev_after = np.sum(contributions * mask, axis=0)
        ev_dict[f"after {last}"] = ev_after

    return ev_dict

def get_intervention_longterm_value(diff_surv, n_years):
    """n_years: array (n_sims,). Returns array (n_sims,)."""
    # Per-sim indexing: diff_surv[n_years[j]-1, j]
    sim_idx = np.arange(n_sims)
    diff_n = diff_surv[n_years - 1, sim_idx]  # (n_sims,)
    cond_V_earth = get_conditional_future_value_on_earth(n_years)

    # â”€â”€ branch: no cubic growth â”€â”€
    ev_no_cubic = diff_n * cond_V_earth

    # â”€â”€ branch: cubic, T_c > n_years â”€â”€
    cV_Ts2 = get_conditional_future_value_stars_to_Ts2()
    cV_Th3 = get_conditional_future_value_stars_to_Th3()
    p_Tc2 = np.exp(-r_inf * (T_c - n_years))
    ev_cubic_late = diff_n * (cond_V_earth + p_Tc2 * (cV_Ts2 + cV_Th3))

    # â”€â”€ branch: cubic, T_c <= n_years â”€â”€
    # exp factor adjusts stellar integrals from T_c-based to n_years-based discounting
    # but must NOT be applied to earth term
    cV_Ts3 = get_conditional_future_value_stars_to_Ts3(n_years)
    exp_adj = np.exp(-r_inf * (T_c - n_years))  # >1 when T_c < n_years
    ev_cubic_early = diff_n * (cond_V_earth + exp_adj * (cV_Ts3 + cV_Th3))

    # select per-sim
    ev_cubic = np.where(T_c > n_years, ev_cubic_late, ev_cubic_early)
    return np.where(cubic_growth, ev_cubic, ev_no_cubic)

def get_total_value_intervention(diff_surv, v_arr, n_years, to_print=False):
    """n_years: array (n_sims,). Returns dict whose values are all (n_sims,)."""
    ev_short = get_intervention_shortterm_value(diff_surv, v_arr, n_years)
    ev_long = get_intervention_longterm_value(diff_surv, n_years)

    ev_dict = dict(ev_short)

    last = periods_value[-1]
    after_key = f"after {last}"
    if after_key in ev_short:
        ev_dict[f"Expected Value after {last}"] = ev_short[after_key] + ev_long

    # Total: sum all unique contributions
    total = np.zeros(n_sims)
    for i in range(len(periods_value) - 1):
        lo, hi = periods_value[i], periods_value[i + 1]
        total += ev_dict[f"{lo} to {hi}"]
    if after_key in ev_short:
        total += ev_short[after_key]
    total += ev_long
    ev_dict["Total Value"] = total

    if to_print:
        for k, v in ev_dict.items():
            print(f"  {k}: {v}")

    return ev_dict


## Value of intervention (QALYs per $1M)

In [57]:
## These outputs can be taken from here and put into the model, to be risk-weighted and discounted 

print("\n=== Intervention EV (per sim), by years when effect occurs ===")
EV_dict = get_total_value_intervention(diff_in_survival, value_array, num_years_per_sim, to_print=True)

print("\n=== QALYs Per $1M (per sim), by time when effect occurs ===")
EV_dict_per_M = {}
for k, v in EV_dict.items():
    key = k + " per $1M"
    EV_dict_per_M[key] = v / budget * 1e6
    print(f"  {key}: {EV_dict_per_M[key]}")


=== Intervention EV (per sim), by years when effect occurs ===
  0 to 5: [ 1.74381542e+02 -4.23936820e+00  1.23716364e+04]
  5 to 10: [ 4.73630311e+02 -3.55240521e+01  6.48246690e+04]
  10 to 20: [ 1.88777912e+03 -1.48936137e+02  3.18025752e+05]
  20 to 100: [ 2.29964203e+04 -1.30787690e+03  8.50002506e+06]
  100 to 500: [ 1.47974383e+05 -7.73111565e+03  6.99991565e+07]
  after 500: [ 1.89652939e+04 -5.96386941e+03  3.09435169e+07]
  Expected Value after 500: [ 3.93263608e+09 -1.18525765e+09  1.32251211e+35]
  Total Value: [ 3.93280959e+09 -1.18526688e+09  1.32251211e+35]

=== QALYs Per $1M (per sim), by time when effect occurs ===
  0 to 5 per $1M: [ 1.74381542e+01 -4.23936820e-01  1.23716364e+03]
  5 to 10 per $1M: [ 4.73630311e+01 -3.55240521e+00  6.48246690e+03]
  10 to 20 per $1M: [ 1.88777912e+02 -1.48936137e+01  3.18025752e+04]
  20 to 100 per $1M: [ 2.29964203e+03 -1.30787690e+02  8.50002506e+05]
  100 to 500 per $1M: [ 1.47974383e+04 -7.73111565e+02  6.99991565e+06]
  after 5